# Where the 259 come from

Compare the decision list with the cryosleep rule. They disagree only when step 3 or step 4 matches, because those two steps predict transported for someone the cryosleep rule predicted to stay.

Step 3 matches 429 passengers: 406 awake, and 23 whose cryosleep cell is blank. Of the 429, 291 were transported and 138 stayed. Changing the prediction from "stayed" to "transported" gains 291 and loses 138.

Step 4 matches 210 passengers: 171 awake, and 39 with a blank cryosleep cell. Of the 210, 158 were transported and 52 stayed. Maham is one of those 52.

$$
\begin{aligned}
(291 - 138) + (158 - 52) &= 153 + 106 = 259 \\
6244 + 259 &= 6503
\end{aligned}
$$

The 806 children of age at most 12 split cleanly around these steps: 377 are already caught by cryosleep, and the other $806 - 377 = 429$ are exactly step 3.

Two further steps are easy to notice and thinner than these.

Among the passengers who reach step 5, starboard side `S` is 492 transported of 908. Predicting transported for those 908, and for nobody else new, changes the full-manifest count by

$$
492 - (908 - 492) = 492 - 416 = 76,
$$

from 6503 to $6503 + 76 = 6579$.

Among the same leftover passengers, destination `55 Cancri e` is 148 transported of 273. That experiment changes the count by

$$
148 - (273 - 148) = 23,
$$

from 6503 to 6526.

Both experiments are real increments on the full manifest. Both bands sit near one half: 492 of 908, and 148 of 273. The list this path reports keeps steps 1 to 5, whose active changes are 291 of 429 and 158 of 210. The next lesson asks whether the thin increments survive a group holdout.


In [2]:
# Locate the contest tables beside this notebook, or under the course root.
from pathlib import Path
import csv

def manifest_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "14-Spaceship Titanic" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "sample_submission.csv").exists():
            return direct
    raise FileNotFoundError(name)

def read_manifest(name):
    with manifest_path(name).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

train = read_manifest("train.csv")
test = read_manifest("test.csv")
submission = read_manifest("sample_submission.csv")
print(len(train), len(test), len(submission))
print(train[0]["PassengerId"], train[0]["Name"])


8693 4277 4277
0001_01 Maham Ofracculy


In [3]:
# Whole numbers are stored as decimals, such as 39.0.
def whole(text):
    return int(float(text))

def luxury(row):
    columns = ("RoomService", "Spa", "VRDeck")
    if any(row[column] == "" for column in columns):
        return None
    return sum(whole(row[column]) for column in columns)

def cryo_rule(row):
    return row["CryoSleep"] == "True"

def decision_list(row):
    # Step 1. Cryosleep.
    if row["CryoSleep"] == "True":
        return True
    # Step 2. Awake, and the luxury bill is at least 500.
    bill = luxury(row)
    if row["CryoSleep"] == "False" and bill is not None and bill >= 500:
        return False
    # Step 3. Age at most 12.
    if row["Age"] != "" and whole(row["Age"]) <= 12:
        return True
    # Step 4. Europa, deck B or deck C.
    if row["Cabin"] != "":
        deck = row["Cabin"].split("/")[0]
        if row["HomePlanet"] == "Europa" and deck in ("B", "C"):
            return True
    # Step 5. Everyone else stays.
    return False

print("rules ready", len(train))


rules ready 8693


In [4]:
# Ledger of the two flips, the children, and the two thin experiments.
def step(row):
    if row["CryoSleep"] == "True":
        return 1
    bill = luxury(row)
    if row["CryoSleep"] == "False" and bill is not None and bill >= 500:
        return 2
    if row["Age"] != "" and whole(row["Age"]) <= 12:
        return 3
    if row["Cabin"] != "":
        deck = row["Cabin"].split("/")[0]
        if row["HomePlanet"] == "Europa" and deck in ("B", "C"):
            return 4
    return 5

def tally(step_number):
    rows = [row for row in train if step(row) == step_number]
    transported = sum(row["Transported"] == "True" for row in rows)
    awake = sum(row["CryoSleep"] == "False" for row in rows)
    blank = sum(row["CryoSleep"] == "" for row in rows)
    return transported, len(rows), awake, blank

step_3 = tally(3)
step_4 = tally(4)
print("step 3", step_3)
print("step 4", step_4)
assert step_3 == (291, 429, 406, 23)
assert step_4 == (158, 210, 171, 39)
assert (291 - 138) + (158 - 52) == 259
children = [
    row for row in train if row["Age"] != "" and whole(row["Age"]) <= 12
]
cryo_children = sum(row["CryoSleep"] == "True" for row in children)
print("children", len(children), "cryo", cryo_children)
assert (len(children), cryo_children) == (806, 377)
assert 806 - 377 == 429

leftover = [row for row in train if step(row) == 5]

def side(row):
    if row["Cabin"] == "":
        return ""
    return row["Cabin"].split("/")[-1]

starboard = [row for row in leftover if side(row) == "S"]
cancri = [row for row in leftover if row["Destination"] == "55 Cancri e"]
starboard_moved = sum(row["Transported"] == "True" for row in starboard)
cancri_moved = sum(row["Transported"] == "True" for row in cancri)
print("starboard", starboard_moved, len(starboard))
print("cancri", cancri_moved, len(cancri))
assert (starboard_moved, len(starboard)) == (492, 908)
assert (cancri_moved, len(cancri)) == (148, 273)
assert 6503 + (492 - 416) == 6579
assert 6503 + (148 - 125) == 6526


step 3 (291, 429, 406, 23)
step 4 (158, 210, 171, 39)
children 806 cryo 377
starboard 492 908
cancri 148 273


## Pitfall

A net of $+76$ on the full manifest can be a band of 492 against 416. That is a majority, and it is close to a tie. Folding it into the reported list would make Maham's kind of audit harder for a gain that the holdout still has to confirm.

## Summary

Step 3 contributes $291 - 138$. Step 4 contributes $158 - 52$. The sum is 259. Starboard-on-the-leftover reaches 6579. Cancri-on-the-leftover reaches 6526. The reported list stays at 6503.
